# Extracción y limpieza de datos con un proceso ETL
## Inversión pública en ciberseguridad registrada en SECOP II (2021–2025)

**Ciencia de Datos con Python (IGS38) — Caso de estudio · Entrega 1: fuentes de datos y manipulación con Pandas (RA1 y RA2)**

**Pregunta del proyecto:** ¿cómo se distribuye la inversión pública en ciberseguridad registrada en SECOP II entre entidades, territorios y tipos de servicio, y cambió después de los incidentes de ransomware de 2023 que afectaron a entidades del Estado?

El conjunto **SECOP II · Contratos Electrónicos** (`jbjy-vk9h`), publicado por Colombia Compra Eficiente en datos.gov.co, tiene una fila por contrato firmado en la plataforma: entidad compradora, proveedor, objeto, modalidad, fechas y valor. En este cuaderno extraemos **solo los contratos de ciberseguridad** mediante la API del portal y los llevamos hasta un conjunto **limpio, validado y documentado**, listo para integrarlo con otras fuentes en la Entrega 2.

> **Cómo leer este ejemplo.** Las decisiones y su justificación están escritas; las **cifras** salen al ejecutar las celdas contra la API. Cuando el texto dice *"observa"*, se refiere a la salida de la celda anterior. Tu equipo debe redactar sus propias conclusiones a partir de sus propias salidas.

## ¿Por qué los datos de SECOP están "sucios"?

Igual que con el dengue, los problemas tienen causas conocidas:

- **Miles de digitadores.** Cada entidad pública registra sus contratos. El objeto del contrato es **texto libre**: el mismo servicio aparece escrito de cientos de formas, con y sin tildes, en mayúsculas o minúsculas.
- **Formatos de la plataforma.** La API entrega **todo como texto**: los valores en pesos, las fechas ISO con hora (`2023-09-14T00:00:00.000`) y los códigos del clasificador UNSPSC, que pueden traer un prefijo (por ejemplo `V1.43233200`).
- **Identificadores inconsistentes.** El NIT puede venir con o sin dígito de verificación, y una misma entidad puede aparecer con nombres distintos.
- **Marcadores de faltante escritos como texto**, como `No Definido`, que `pandas` no reconoce como faltantes.
- **Filtros por palabra que traen basura.** Buscar `SIEM` (un sistema de monitoreo de seguridad) también trae `SIEMBRA` y `SIEMPRE`. Decidir qué contratos son de ciberseguridad es, en sí mismo, una decisión de limpieza.

Limpiar no es "arreglar a ojo": cada decisión debe estar **justificada**, **verificada** (antes y después) y **documentada**.

## Proceso ETL

| Etapa | Qué hacemos en este cuaderno |
|---|---|
| **E — Extraer** | Documentar la fuente (RA1) y descargar los contratos por la **API Socrata**, filtrando en el servidor. |
| **T — Transformar** | Explorar, eliminar lo que sobra, corregir texto, estructurar tipos, asignar DIVIPOLA y completar faltantes. |
| **L — Cargar** | Guardar el conjunto limpio (Parquet y CSV) y el reporte de calidad. |

A diferencia del cuaderno del dengue, aquí **no** partimos de un archivo descargado a mano: el cuaderno descarga los datos él mismo. Así cualquiera puede reproducirlo, y el repositorio no necesita guardar archivos pesados (norma de entrega del curso).

## Ruta del cuaderno

1. **Extraer**: ficha de la fuente, columnas, filtro en el servidor y descarga paginada.
2. **Transformar**
   - Exploración inicial (diagnóstico).
   - T1. Eliminar: duplicados, contratos que no son inversión, falsos positivos del filtro y columnas irrelevantes.
   - T2. Corregir: espacios, marcadores, NIT, texto y categorías.
   - T3. Estructurar: fechas, valores, UNSPSC, rangos, tipo de servicio, DIVIPOLA y reglas de consistencia.
   - T4. Completar los datos que faltan.
   - Agregaciones de control (`groupby`, `pivot_table`, `resample`).
3. **Cargar** el conjunto limpio.
4. **Validación final** sobre el archivo cargado.
5. **Reporte de calidad** y conclusiones.

In [ ]:
import json
import os
import re
import unicodedata
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 80)

### Configuración

Todo lo que podría cambiar (identificadores de los conjuntos, periodo, carpetas) se define **una sola vez** aquí arriba.

- `SOCRATA_APP_TOKEN`: la API funciona sin token, pero con límites de velocidad más bajos. Crea uno gratis en datos.gov.co (*Perfil → Configuración del desarrollador*) y guárdalo como variable de entorno; **nunca** lo escribas en el cuaderno ni lo subas al repositorio.
- Los datos descargados se guardan en `data/raw/`. Si el archivo ya existe, el cuaderno lo reutiliza en lugar de volver a descargar. Para actualizar, borra esa carpeta. `data/` va en el `.gitignore`.

In [ ]:
DOMINIO = "www.datos.gov.co"
DATASET_SECOP = "jbjy-vk9h"      # SECOP II - Contratos Electrónicos (Colombia Compra Eficiente)
DATASET_DIVIPOLA = "gdxc-w37w"   # DIVIPOLA - Códigos de municipios (DANE)
APP_TOKEN = os.environ.get("SOCRATA_APP_TOKEN")

FECHA_INICIO = "2021-01-01"
FECHA_FIN = "2025-12-31"

RAW = Path("data/raw")
PROCESSED = Path("data/processed")
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

### Herramientas de apoyo

Las mismas del cuaderno del dengue, más dos nuevas:

- `cambios` y `registrar(...)`: registran cada modificación para la tabla resumen.
- `n_distintos(antes, despues)`: cuenta cuántos valores cambiaron (verificación antes y después).
- `normalizar_texto(...)`: mayúsculas, sin tildes (conservando la Ñ) y sin espacios sobrantes. La usamos para **comparar** textos, no para mostrarlos.
- `cliente_socrata()`: abre la conexión con la API solo cuando hace falta descargar algo.

In [ ]:
cambios = []

def registrar(etapa, operacion, tipo, cantidad, detalle=""):
    cambios.append({"etapa": etapa, "operacion": operacion, "tipo": tipo,
                    "cantidad": int(cantidad), "detalle": detalle})

def n_distintos(antes, despues):
    ambos_nulos = antes.isna() & despues.isna()
    return int(((antes != despues) & ~ambos_nulos).sum())

def normalizar_texto(texto):
    if pd.isna(texto):
        return texto
    descompuesto = unicodedata.normalize("NFD", str(texto).upper())
    sin_marcas = re.sub(r"(?<!N)[̀-ͯ]|(?<=N)[̀-̂̄-ͯ]", "", descompuesto)
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", sin_marcas)).strip()

def cliente_socrata():
    from sodapy import Socrata
    return Socrata(DOMINIO, APP_TOKEN, timeout=600)

---
# 1. EXTRAER

## 1.1 Ficha de la fuente (RA1)

El RA1 pide que la fuente esté **documentada**: entidad, enlace, fecha de corte, frecuencia de actualización, diccionario de datos y licencia. No hace falta copiarlos a mano: la API publica los **metadatos** del conjunto en `get_metadata`. Los guardamos junto con la **fecha de consulta**, porque los datos de SECOP cambian todos los días y la entrega debe poder decir *qué versión* se analizó.

In [ ]:
def obtener_metadatos(dataset_id):
    ruta = RAW / f"metadatos_{dataset_id}.json"
    if ruta.exists():
        return json.loads(ruta.read_text(encoding="utf-8"))
    with cliente_socrata() as cliente:
        meta = cliente.get_metadata(dataset_id)
    meta["_fecha_consulta"] = datetime.now().isoformat(timespec="seconds")
    ruta.write_text(json.dumps(meta, ensure_ascii=False, indent=1), encoding="utf-8")
    return meta

def ficha(meta, dataset_id):
    campos = meta.get("metadata", {}).get("custom_fields", {})
    frecuencia = next((v for bloque in campos.values() for k, v in bloque.items()
                       if "frecuencia" in k.lower()), "No publicada")
    actualizado = meta.get("rowsUpdatedAt") or meta.get("dataUpdatedAt")
    if isinstance(actualizado, (int, float)):
        actualizado = datetime.fromtimestamp(actualizado).isoformat(timespec="seconds")
    return pd.Series({
        "nombre": meta.get("name"),
        "entidad": meta.get("attribution", "No publicada"),
        "enlace": f"https://{DOMINIO}/d/{dataset_id}",
        "última actualización (fecha de corte)": actualizado,
        "frecuencia de actualización": frecuencia,
        "licencia": (meta.get("license") or {}).get("name", "No publicada"),
        "fecha de consulta": meta.get("_fecha_consulta"),
        "número de columnas": len(meta.get("columns", [])),
    })

meta_secop = obtener_metadatos(DATASET_SECOP)
ficha(meta_secop, DATASET_SECOP).to_frame("SECOP II · Contratos Electrónicos")

### Diccionario de datos

Cada columna de la API trae su nombre técnico (`fieldName`, el que usamos en el código), su nombre legible, su tipo y una descripción. Este es el **diccionario de datos** que pide la entrega; lo guardamos como CSV.

In [ ]:
diccionario = pd.DataFrame(meta_secop.get("columns", []))
diccionario = diccionario.reindex(columns=["fieldName", "name", "dataTypeName", "description"])
diccionario.to_csv(PROCESSED / "diccionario_secop.csv", index=False, encoding="utf-8-sig")
diccionario.head(20)

### Valoración de la calidad de la fuente

La rúbrica del RA1 pide **valorar** la calidad, no solo describir la fuente. Antes de limpiar, dejamos escrito qué esperamos; al final del cuaderno lo contrastamos con lo que encontramos.

| Dimensión | Valoración inicial | Cómo se verifica en este cuaderno |
|---|---|---|
| Pertinencia | Alta: registra la compra pública con su objeto y su valor | — |
| Cobertura | Parcial: solo lo contratado en SECOP II; no incluye SECOP I ni compras dentro de contratos generales de TI | Limitación declarada |
| Oportunidad | Alta: actualización frecuente (ver ficha) | Fecha de corte registrada |
| Completitud | Por medir | % de nulos y marcadores por columna (2.0) |
| Consistencia | Por medir | Reglas de consistencia (T3.8) y NIT con un único nombre (T2.6) |
| Exactitud del filtro | Por medir | Falsos positivos del filtro por palabras (T1.3) |

## 1.2 Columnas que vamos a extraer

El conjunto tiene más de 70 columnas; pedimos solo las que responden la pregunta. Antes de descargar comprobamos que **todas existan** en el diccionario: Colombia Compra ha cambiado nombres de columnas entre versiones, y una columna mal escrita hace fallar la consulta.

In [ ]:
COLUMNAS = [
    # identificación y trazabilidad
    "id_contrato", "urlproceso",
    # entidad compradora
    "nombre_entidad", "nit_entidad", "departamento", "ciudad", "orden", "sector",
    # contrato
    "estado_contrato", "codigo_de_categoria_principal", "objeto_del_contrato",
    "tipo_de_contrato", "modalidad_de_contratacion",
    # proveedor
    "proveedor_adjudicado", "documento_proveedor",
    # tiempo y valor
    "fecha_de_firma", "fecha_de_inicio_del_contrato", "fecha_de_fin_del_contrato",
    "valor_del_contrato",
]

disponibles = set(diccionario["fieldName"])
faltan = [c for c in COLUMNAS if c not in disponibles]
print("Columnas que no existen en el diccionario:", faltan or "ninguna")
COLUMNAS = [c for c in COLUMNAS if c in disponibles]

> Si la lista anterior no está vacía, busca en `diccionario` el nombre nuevo de esa columna (por ejemplo con `diccionario[diccionario["name"].str.contains("firma", case=False)]`) y corrige `COLUMNAS`.

## 1.3 Filtro en el servidor (SoQL)

SECOP II tiene millones de contratos. Descargarlos todos para luego quedarnos con unos miles sería lento y desperdiciaría la API. La guía pide **filtrar en el servidor**: la consulta viaja con una cláusula `$where` escrita en **SoQL**, el lenguaje de consulta de Socrata (muy parecido a SQL).

El filtro del servidor es **amplio a propósito**: trae todo lo que *podría* ser ciberseguridad. El filtro fino, con expresiones regulares, se hace después en `pandas` (T1.3), donde podemos ver y justificar cada exclusión.

| Señal | Valores |
|---|---|
| Código UNSPSC | `43233200` software de seguridad y protección · `43222500` equipo de seguridad de red · `81111801` seguridad de computadores, redes o internet |
| Palabras en el objeto | CIBER, SEGURIDAD INFORMATICA, SEGURIDAD DIGITAL, SEGURIDAD DE LA INFORMACION, FIREWALL, ANTIVIRUS, ETHICAL HACKING, PENTEST, VULNERABILIDAD, SIEM, ISO 27001 |

> Confirma los códigos en el **Clasificador de Bienes y Servicios** de Colombia Compra Eficiente. En SoQL, `upper()` pasa el texto a mayúsculas; buscamos raíces sin tilde (`INFORM`, `INFORMACI`) para no depender de cómo se escribió la tilde.

In [ ]:
CODIGOS_UNSPSC = ["43233200", "43222500", "81111801"]
RAICES_OBJETO = ["CIBER", "SEGURIDAD INFORM", "SEGURIDAD DIGITAL", "SEGURIDAD DE LA INFORMACI",
                 "FIREWALL", "ANTIVIRUS", "ETHICAL HACKING", "PENTEST", "VULNERABILIDAD",
                 "SIEM", "27001"]

condicion_texto = " OR ".join(f"upper(objeto_del_contrato) like '%{r}%'" for r in RAICES_OBJETO)
condicion_codigo = " OR ".join(f"codigo_de_categoria_principal like '%{c}%'" for c in CODIGOS_UNSPSC)

WHERE = (f"fecha_de_firma between '{FECHA_INICIO}T00:00:00' and '{FECHA_FIN}T23:59:59' "
         f"AND ({condicion_texto} OR {condicion_codigo})")
print(WHERE)

## 1.4 Descarga paginada

La API entrega los resultados por **páginas** (`limit` filas a partir de `offset`). Pedimos páginas de 50 000 filas, **ordenadas** por `id_contrato` para que la paginación sea estable, hasta que una página llegue incompleta (con menos filas que el lote): así no hacemos una petición extra que en esta API puede tardar varios minutos.

### ¿Por qué todo como texto?

La API ya entrega cada valor como texto. Lo mantenemos así al guardar la copia en bruto (`df_raw`) y convertimos cada columna a su tipo en la etapa T3, igual que hicimos con el dengue: así los códigos no pierden ceros y podemos medir qué se perdió en cada conversión.

In [ ]:
def descargar(dataset_id, select, where=None, order=None, lote=50_000):
    filas, offset = [], 0
    with cliente_socrata() as cliente:
        while True:
            pagina = cliente.get(dataset_id, select=select, where=where, order=order,
                                 limit=lote, offset=offset)
            if not pagina:
                break
            filas.extend(pagina)
            offset += lote
            print(f"  {len(filas):,} filas descargadas")
            if len(pagina) < lote:   # página incompleta: ya no quedan más filas
                break
    return pd.DataFrame.from_records(filas).astype("string")

ruta_raw = RAW / "secop_ciberseguridad_raw.parquet"

if ruta_raw.exists():
    df_raw = pd.read_parquet(ruta_raw)
else:
    df_raw = descargar(DATASET_SECOP, select=", ".join(COLUMNAS), where=WHERE, order="id_contrato")
    df_raw.to_parquet(ruta_raw, index=False)

df_raw = df_raw.astype("string").reindex(columns=COLUMNAS)
df = df_raw.copy()

---
# 2. TRANSFORMAR

## 2.0 Exploración inicial

Antes de cambiar nada hay que **conocer** los datos.

### Dimensiones y primeras filas

In [ ]:
df.shape

In [ ]:
df.head()

### Estructura y tipos: `info()`

Todo es texto (`string`). Observa los conteos de no nulos: la API **omite** los campos vacíos, así que esas celdas llegan como faltantes.

In [ ]:
df.info()

### Resumen: `describe()`

Con texto, `describe()` muestra `count`, `unique`, `top` y `freq`. Fíjate en tres pistas: columnas con muy pocos valores distintos (candidatas a categoría), un `top` que es un marcador como `No Definido`, y si `id_contrato` tiene tantos valores únicos como filas.

In [ ]:
df.describe().T

### Nulos, celdas en blanco y marcadores

Contamos tres cosas por columna:

- `nulos`: faltantes que `pandas` reconoce.
- `solo_espacios`: celdas que parecen tener dato pero están en blanco.
- `marcadores`: textos que significan "no hay dato" (`No Definido`, `Sin Descripcion`, `N/A`…).

In [ ]:
MARCADORES = {"", "NO DEFINIDO", "SIN DESCRIPCION", "N/A", "NA", "NULL", "-"}

def es_marcador(serie):
    return serie.map(normalizar_texto, na_action="ignore").isin(MARCADORES).fillna(False)

calidad_inicial = pd.DataFrame({
    "nulos": df.isna().sum(),
    "pct_nulos": (df.isna().mean() * 100).round(2),
    "solo_espacios": df.apply(lambda s: s.str.strip().eq("").sum()),
    "marcadores": df.apply(lambda s: es_marcador(s).sum()),
})
calidad_inicial["pct_sin_dato"] = ((calidad_inicial["nulos"] + calidad_inicial["marcadores"])
                                   / len(df) * 100).round(2)
calidad_inicial.sort_values("pct_sin_dato", ascending=False)

### Duplicados

En SECOP la llave natural es `id_contrato`. Contamos filas idénticas en **todas** las columnas y filas que repiten el **identificador**.

In [ ]:
pd.Series({
    "duplicados exactos": df.duplicated().sum(),
    "id_contrato repetido": df["id_contrato"].duplicated().sum(),
    "id_contrato faltante": df["id_contrato"].isna().sum(),
})

### Hallazgos del diagnóstico

Esta tabla es la que tu equipo llena **con sus propias cifras** a partir de las salidas anteriores. Aquí aparecen los problemas que se deben buscar en SECOP II.

| Problema a buscar | Dónde mirarlo | Operación |
|---|---|---|
| Contratos repetidos | `id_contrato` repetido | T1 Eliminar |
| Contratos cancelados o anulados | `estado_contrato` | T1 Eliminar |
| Falsos positivos del filtro por palabras | `objeto_del_contrato` (`SIEMBRA`, `SIEMPRE`, vigilancia privada) | T1 Eliminar |
| Espacios sobrantes y marcadores como texto | `ciudad`, `departamento`, `objeto_del_contrato` | T2 Corregir |
| NIT con puntos, guiones o dígito de verificación | `nit_entidad` | T2 Corregir |
| Una entidad con varios nombres | `nit_entidad` vs `nombre_entidad` | T2 Corregir |
| Fechas y valores guardados como texto | `fecha_de_*`, `valor_del_contrato` | T3 Estructurar |
| Código UNSPSC con prefijo | `codigo_de_categoria_principal` | T3 Estructurar |
| Municipio como nombre, sin código DIVIPOLA | `ciudad` + `departamento` | T3 Estructurar |
| Municipio faltante | `ciudad = No Definido` | T4 Completar |

Los nombres de columna de la API ya vienen en *snake_case*; lo verificamos en lugar de suponerlo:

In [ ]:
assert all(re.fullmatch(r"[a-z0-9_]+", c) for c in df.columns)

---
## T1. Eliminar los datos no deseados

### T1.1 Contratos repetidos — Excel: Quitar duplicados

Primero miramos los repetidos para entender si son el mismo contrato:

In [ ]:
repetidos = df[df["id_contrato"].duplicated(keep=False)].sort_values("id_contrato")
repetidos[["id_contrato", "nombre_entidad", "fecha_de_firma", "valor_del_contrato", "estado_contrato"]].head(10)

Si dos filas tienen el mismo `id_contrato`, describen el mismo contrato (por ejemplo, una versión anterior y una actualizada). Conservamos **una** por identificador: la de fecha de firma más reciente. Si en tu salida difieren en el valor, discútelo antes de borrar.

In [ ]:
filas_antes = len(df)

# Excel: Datos → Quitar duplicados (sobre la columna id_contrato)
df = (df.sort_values(["id_contrato", "fecha_de_firma"])
        .drop_duplicates(subset="id_contrato", keep="last")
        .reset_index(drop=True))

pd.DataFrame({
    "antes": [filas_antes, df_raw["id_contrato"].duplicated().sum()],
    "despues": [len(df), df["id_contrato"].duplicated().sum()],
}, index=["filas", "id_contrato repetido"])

In [ ]:
registrar("T1 Eliminar", "Contratos con id_contrato repetido", "Filas eliminadas", filas_antes - len(df))

### T1.2 Contratos que no representan inversión

Un contrato cancelado, anulado o en borrador no se ejecutó: sumarlo inflaría la inversión. Revisamos los estados con `value_counts()` (en Excel, abrir el **filtro** de la columna):

In [ ]:
df["estado_contrato"].value_counts(dropna=False)

In [ ]:
ESTADOS_EXCLUIDOS = {"CANCELADO", "ANULADO", "BORRADOR"}

estado_cmp = df["estado_contrato"].map(normalizar_texto, na_action="ignore")
a_excluir = estado_cmp.isin(ESTADOS_EXCLUIDOS).fillna(False)

filas_antes = len(df)
df = df[~a_excluir].reset_index(drop=True)

pd.Series({"filas antes": filas_antes, "filas eliminadas": int(a_excluir.sum()), "filas después": len(df)})

In [ ]:
registrar("T1 Eliminar", "Contratos cancelados, anulados o en borrador", "Filas eliminadas",
          filas_antes - len(df), ", ".join(sorted(ESTADOS_EXCLUIDOS)))

### T1.3 Falsos positivos del filtro por palabras

El filtro del servidor buscó **subcadenas**. Eso trae contratos que no son de ciberseguridad. Ahora aplicamos la regla fina, con **límites de palabra** (`\b` en una expresión regular: `\bSIEM\b` encuentra `SIEM` pero no `SIEMBRA`).

Un contrato se **conserva** si cumple al menos una de dos señales (código UNSPSC o palabra clave) y **no** contiene un término de exclusión:

| Lista | Contenido | Por qué |
|---|---|---|
| Inclusión | ciberseguridad, seguridad informática/digital/de la información, firewall, antivirus, EDR, SIEM, SOC, ethical hacking, pentest, vulnerabilidades, ISO 27001, SGSI | Servicios de seguridad informática |
| Exclusión | vigilancia, guardas, escoltas, seguridad privada, seguridad y salud en el trabajo | "Seguridad" en otro sentido |

Comparamos sobre una versión normalizada del objeto (mayúsculas, sin tildes) que **no** reemplaza la columna original.

In [ ]:
INCLUIR = (r"\bCIBER|\bSEGURIDAD (?:INFORMATICA|DIGITAL|DE LA INFORMACION)\b|\bFIREWALL|\bANTIVIRUS\b"
           r"|\bEDR\b|\bSIEM\b|\bSOC\b|\bETHICAL HACKING\b|\bPENTEST|\bPRUEBAS DE PENETRACION\b"
           r"|\bVULNERABILIDAD|\bISO ?27001\b|\bSGSI\b")
EXCLUIR = (r"\bVIGILANCIA\b|\bGUARDAS?\b|\bESCOLTAS?\b|\bSEGURIDAD PRIVADA\b"
           r"|\bSEGURIDAD Y SALUD\b|\bSST\b")

objeto_cmp = df["objeto_del_contrato"].map(normalizar_texto, na_action="ignore")
codigo_cmp = df["codigo_de_categoria_principal"].str.extract(r"(\d{8})", expand=False)

por_codigo = codigo_cmp.isin(CODIGOS_UNSPSC).fillna(False)
por_texto = objeto_cmp.str.contains(INCLUIR, regex=True, na=False)
excluido = objeto_cmp.str.contains(EXCLUIR, regex=True, na=False)
es_ciber = (por_codigo | por_texto) & ~excluido

pd.Series({
    "solo por código UNSPSC": int((por_codigo & ~por_texto).sum()),
    "solo por palabra clave": int((por_texto & ~por_codigo).sum()),
    "por ambas señales": int((por_codigo & por_texto).sum()),
    "eliminados por término de exclusión": int(((por_codigo | por_texto) & excluido).sum()),
    "sin ninguna señal (falsos positivos del servidor)": int((~por_codigo & ~por_texto).sum()),
    "se conservan": int(es_ciber.sum()),
})

**Verificación manual.** La estadística no basta: leemos una muestra de lo que **eliminamos**. Si aparece un contrato que sí es de ciberseguridad, hay que ajustar la regla.

In [ ]:
df.loc[~es_ciber, "objeto_del_contrato"].sample(min(15, int((~es_ciber).sum())), random_state=1).to_frame()

Y una muestra de lo que **conservamos**, para estimar los falsos positivos que quedan:

In [ ]:
df.loc[es_ciber, "objeto_del_contrato"].sample(min(15, int(es_ciber.sum())), random_state=1).to_frame()

In [ ]:
filas_antes = len(df)
df = df[es_ciber].reset_index(drop=True)

registrar("T1 Eliminar", "Falsos positivos del filtro por palabras", "Filas eliminadas",
          filas_antes - len(df), "regla: (UNSPSC o palabra clave) y sin término de exclusión")

> **Para discutir:** la regla solo mira el objeto y el código principal. Un contrato de "licencias de software" que incluye un antivirus quedaría por fuera. Con 100 contratos revisados a mano se puede estimar la **precisión** del filtro; en la Parte 3 podría reemplazarse por un clasificador de texto.

### T1.4 Columnas irrelevantes

Como seleccionamos las columnas en la consulta, no esperamos columnas vacías, pero lo comprobamos. Una columna **constante** tampoco aporta información.

In [ ]:
columnas_vacias = df.columns[df.isna().all()].tolist()
columnas_constantes = [c for c in df.columns if df[c].nunique(dropna=True) == 1]

pd.Series({"vacías": columnas_vacias, "constantes": columnas_constantes})

In [ ]:
columnas_a_eliminar = columnas_vacias + columnas_constantes
df = df.drop(columns=columnas_a_eliminar)

registrar("T1 Eliminar", "Columnas vacías o constantes", "Columnas eliminadas",
          len(columnas_a_eliminar), ", ".join(columnas_a_eliminar))

---
## T2. Corregir los datos incorrectos (texto)

### T2.1 Espacios sobrantes — Excel: ESPACIOS / TRIM

In [ ]:
columnas_texto = df.columns.tolist()

def contar_espacios(data):
    extremos = sum(((data[c] != data[c].str.strip()) & data[c].notna()).sum() for c in columnas_texto)
    dobles = sum(data[c].str.strip().str.contains(r"\s{2,}", na=False).sum() for c in columnas_texto)
    return int(extremos), int(dobles)

extremos_antes, dobles_antes = contar_espacios(df)
copia_antes = df[columnas_texto].copy()

for c in columnas_texto:
    # Excel: =ESPACIOS(celda)  /  =TRIM(cell)
    df[c] = df[c].str.strip().str.replace(r"\s+", " ", regex=True)

extremos_despues, dobles_despues = contar_espacios(df)

pd.DataFrame({
    "antes": [extremos_antes, dobles_antes],
    "despues": [extremos_despues, dobles_despues],
}, index=["celdas con espacios en los extremos", "celdas con espacios dobles internos"])

In [ ]:
registrar("T2 Corregir", "ESPACIOS/TRIM: espacios en extremos y dobles internos", "Valores corregidos",
          sum(n_distintos(copia_antes[c], df[c]) for c in columnas_texto))

### T2.2 Marcadores de dato faltante

`No Definido`, las celdas en blanco y similares no son valores: los convertimos en `NaN` (`pd.NA`). Así no aparecerán como una "ciudad" más en los conteos.

In [ ]:
marcadores_antes = int(sum(es_marcador(df[c]).sum() for c in columnas_texto))
nulos_antes = int(df.isna().sum().sum())

for c in columnas_texto:
    df[c] = df[c].mask(es_marcador(df[c]))

pd.DataFrame({
    "antes": [marcadores_antes, nulos_antes],
    "despues": [int(sum(es_marcador(df[c]).sum() for c in columnas_texto)), int(df.isna().sum().sum())],
}, index=["celdas con marcador de faltante", "valores nulos"])

In [ ]:
registrar("T2 Corregir", "Marcadores ('No Definido', vacío, N/A…) convertidos a nulo", "Valores corregidos",
          marcadores_antes)

### T2.3 NIT de la entidad

El NIT es la **llave** con la que en la Entrega 2 cruzaremos con otras fuentes (por ejemplo, FURAG). Si una fuente escribe `890.399.011-3` y la otra `890399011`, el cruce falla. Revisamos su forma con la longitud (Excel: `LARGO`) y los caracteres que no son dígitos:

In [ ]:
largo_nit = df["nit_entidad"].str.replace(r"\D", "", regex=True).str.len()
print("NIT con caracteres no numéricos:", int(df["nit_entidad"].str.contains(r"\D", na=False).sum()))

# Excel: =LARGO(celda) y luego un filtro sobre esa columna
largo_nit.value_counts().sort_index().rename_axis("dígitos").to_frame("contratos")

**Decisión:** dejamos solo dígitos y, si quedan 10, quitamos el último (el **dígito de verificación**). El NIT de una entidad pública tiene 9 dígitos. Lo guardamos como **texto**: como número se perderían los ceros iniciales y nadie suma NIT.

Comprobamos el efecto de la decisión: si es correcta, el número de NIT distintos debe **bajar o mantenerse**, nunca subir.

In [ ]:
nit_antes = df["nit_entidad"].copy()

solo_digitos = df["nit_entidad"].str.replace(r"\D", "", regex=True)
df["nit_entidad"] = solo_digitos.where(solo_digitos.str.len() != 10, solo_digitos.str[:9])

pd.DataFrame({
    "antes": [nit_antes.nunique(), int(nit_antes.str.contains(r"\D", na=False).sum())],
    "despues": [df["nit_entidad"].nunique(), int(df["nit_entidad"].str.contains(r"\D", na=False).sum())],
}, index=["NIT distintos", "NIT con caracteres no numéricos"])

In [ ]:
registrar("T2 Corregir", "NIT: solo dígitos, sin dígito de verificación", "Valores corregidos",
          n_distintos(nit_antes, df["nit_entidad"]))

if "documento_proveedor" in df.columns:
    df["documento_proveedor"] = df["documento_proveedor"].str.replace(r"\D", "", regex=True).replace("", pd.NA)

### T2.4 Texto para comparar y texto para mostrar

El objeto del contrato se queda como lo escribió la entidad (es la evidencia), pero creamos `objeto_norm` en mayúsculas y sin tildes para **buscar** en él. Con departamento y ciudad hacemos dos cosas:

- versiones normalizadas (`departamento_norm`, `ciudad_norm`) para el cruce con DIVIPOLA en T3.7;
- versiones con mayúscula inicial para tablas y gráficos — Excel: **NOMPROPIO**.

In [ ]:
copia_antes = df[["departamento", "ciudad"]].copy()

df["objeto_norm"] = df["objeto_del_contrato"].map(normalizar_texto, na_action="ignore")
df["departamento_norm"] = df["departamento"].map(normalizar_texto, na_action="ignore")
df["ciudad_norm"] = df["ciudad"].map(normalizar_texto, na_action="ignore")

for c in ["departamento", "ciudad"]:
    # Excel: =NOMPROPIO(celda)
    df[c] = df[c].str.title()

pd.DataFrame({"antes": copia_antes["ciudad"].drop_duplicates().head(8).values,
              "despues": df["ciudad"].drop_duplicates().head(8).values,
              "para comparar": df["ciudad_norm"].drop_duplicates().head(8).values})

In [ ]:
registrar("T2 Corregir", "NOMPROPIO en departamento y ciudad", "Valores corregidos",
          sum(n_distintos(copia_antes[c], df[c]) for c in ["departamento", "ciudad"]))
registrar("T2 Corregir", "Columnas normalizadas para comparar (objeto, departamento, ciudad)",
          "Columnas creadas", 3)

### T2.5 Categorías mal escritas — Excel: Filtros

Revisamos las columnas categóricas. Si dos valores solo difieren en mayúsculas o tildes, son la misma categoría.

In [ ]:
columnas_categoricas = ["orden", "sector", "tipo_de_contrato", "modalidad_de_contratacion", "estado_contrato"]
columnas_categoricas = [c for c in columnas_categoricas if c in df.columns]

pd.DataFrame({
    "categorías distintas": {c: df[c].nunique() for c in columnas_categoricas},
    "distintas tras normalizar": {c: df[c].map(normalizar_texto, na_action="ignore").nunique()
                                  for c in columnas_categoricas},
})

In [ ]:
# Excel: abrir el filtro de la columna y revisar la lista de valores
df["modalidad_de_contratacion"].value_counts(dropna=False)

Si en la tabla anterior las dos columnas difieren, unificamos cada variante con la forma **más frecuente** de su versión normalizada. Así no escribimos a mano un diccionario por columna, pero dejamos registrado cuántos valores cambiaron.

In [ ]:
copia_antes = df[columnas_categoricas].copy()

for c in columnas_categoricas:
    clave = df[c].map(normalizar_texto, na_action="ignore")
    forma_frecuente = df.groupby(clave)[c].agg(lambda s: s.value_counts().index[0])
    df[c] = clave.map(forma_frecuente)

registrar("T2 Corregir", "Filtros: variantes de categorías unificadas", "Valores corregidos",
          sum(n_distintos(copia_antes[c], df[c]) for c in columnas_categoricas))

pd.Series({c: df[c].nunique() for c in columnas_categoricas}, name="categorías después")

### T2.6 Una entidad, un nombre

Como en el dengue (cada código, un nombre), comprobamos que cada NIT tenga **un solo** nombre de entidad:

In [ ]:
nombres_por_nit = df.groupby("nit_entidad")["nombre_entidad"].nunique()
nits_con_varios = nombres_por_nit[nombres_por_nit > 1].index

print("NIT con más de un nombre:", len(nits_con_varios))
(df[df["nit_entidad"].isin(nits_con_varios)]
   .groupby(["nit_entidad", "nombre_entidad"]).size().head(12).to_frame("contratos"))

Suelen ser la misma entidad escrita de dos formas (con y sin sigla, con y sin tildes). **Decisión:** conservamos el nombre más frecuente de cada NIT. El NIT, no el nombre, es la llave.

In [ ]:
nombre_antes = df["nombre_entidad"].copy()

nombre_frecuente = df.groupby("nit_entidad")["nombre_entidad"].agg(
    lambda s: s.value_counts().index[0] if s.notna().any() else pd.NA)
df["nombre_entidad"] = df["nit_entidad"].map(nombre_frecuente).fillna(df["nombre_entidad"])

assert (df.groupby("nit_entidad")["nombre_entidad"].nunique() <= 1).all()
registrar("T2 Corregir", "Un nombre por NIT (el más frecuente)", "Valores corregidos",
          n_distintos(nombre_antes, df["nombre_entidad"]))

### T2.7 ¿Aparecieron nuevos duplicados?

In [ ]:
pd.Series({"duplicados exactos": df.duplicated().sum(),
           "id_contrato repetido": df["id_contrato"].duplicated().sum()})

---
## T3. Estructurar los datos con los tipos correctos

### T3.1 Fechas

La API entrega fechas ISO con hora (`2023-09-14T00:00:00.000`). Con `format="ISO8601"`, `pd.to_datetime` las interpreta sin ambigüedad. Con `errors="coerce"` una fecha ilegible queda como `NaT`, por eso comparamos los faltantes antes y después.

In [ ]:
columnas_fecha = ["fecha_de_firma", "fecha_de_inicio_del_contrato", "fecha_de_fin_del_contrato"]

nulos_fecha_antes = df[columnas_fecha].isna().sum()
ejemplo_antes = df[columnas_fecha].iloc[0]

for c in columnas_fecha:
    df[c] = pd.to_datetime(df[c], format="ISO8601", errors="coerce").dt.normalize()

pd.DataFrame({
    "ejemplo_antes": ejemplo_antes, "ejemplo_despues": df[columnas_fecha].iloc[0],
    "nulos_antes": nulos_fecha_antes, "nulos_despues": df[columnas_fecha].isna().sum(),
    "tipo": df[columnas_fecha].dtypes,
})

In [ ]:
nuevos_nulos_fecha = int((df[columnas_fecha].isna().sum() - nulos_fecha_antes).sum())
registrar("T3 Estructurar", "Fechas ISO convertidas a datetime", "Columnas convertidas",
          len(columnas_fecha), f"{nuevos_nulos_fecha} fechas ilegibles quedaron como NaT")

### T3.2 Valor del contrato

El valor es una **cantidad**: lo convertimos a número (`float64`, en pesos corrientes). En la Entrega 2 lo pasaremos a pesos constantes con el IPC.

In [ ]:
valor_texto = df["valor_del_contrato"].copy()
df["valor_del_contrato"] = pd.to_numeric(valor_texto, errors="coerce").astype("float64")

pd.Series({
    "nulos antes": int(valor_texto.isna().sum()),
    "nulos después": int(df["valor_del_contrato"].isna().sum()),
    "textos que no eran números": int(valor_texto.notna().sum() - df["valor_del_contrato"].notna().sum()),
    "tipo": str(df["valor_del_contrato"].dtype),
})

> Si aparecen textos que no eran números, míralos con `valor_texto[df["valor_del_contrato"].isna() & valor_texto.notna()]`. Un separador de miles o un signo `$` se corrige antes de convertir, no se convierte a `NaN` sin más.

### T3.3 Código UNSPSC

El clasificador UNSPSC es **jerárquico**: segmento (2 dígitos), familia (4), clase (6) y producto (8). Extraemos los 8 dígitos y creamos el segmento y la clase. Siguen siendo **texto**: son códigos.

In [ ]:
codigo_antes = df["codigo_de_categoria_principal"].copy()

df["unspsc"] = codigo_antes.str.extract(r"(\d{8})", expand=False)
df["unspsc_segmento"] = df["unspsc"].str[:2]
df["unspsc_clase"] = df["unspsc"].str[:6]

pd.DataFrame({"antes": codigo_antes.head(5).values, "unspsc": df["unspsc"].head(5).values,
              "segmento": df["unspsc_segmento"].head(5).values})

In [ ]:
df["unspsc_segmento"].value_counts().head(10)

In [ ]:
registrar("T3 Estructurar", "UNSPSC: código de 8 dígitos, segmento y clase", "Columnas creadas", 3)
df = df.drop(columns=["codigo_de_categoria_principal"])

### T3.4 Rangos de valores — Excel: MAXIMO / MINIMO

| Variable | Rango posible |
|---|---|
| `valor_del_contrato` | mayor que 0 |
| `fecha_de_firma` | dentro del periodo consultado (2021-01-01 a 2025-12-31) |
| `fecha_de_inicio_del_contrato`, `fecha_de_fin_del_contrato` | entre 2020 y 2035 |

In [ ]:
# Excel: =MIN(rango) y =MAX(rango)
df[["valor_del_contrato"] + columnas_fecha].agg(["min", "max"]).T

In [ ]:
fuera_de_rango = {
    "valor <= 0": df["valor_del_contrato"] <= 0,
    "firma fuera del periodo": ~df["fecha_de_firma"].between(FECHA_INICIO, FECHA_FIN),
    "inicio fuera de 2020-2035": ~df["fecha_de_inicio_del_contrato"].between("2020-01-01", "2035-12-31")
                                 & df["fecha_de_inicio_del_contrato"].notna(),
    "fin fuera de 2020-2035": ~df["fecha_de_fin_del_contrato"].between("2020-01-01", "2035-12-31")
                              & df["fecha_de_fin_del_contrato"].notna(),
}
pd.Series({k: int(v.fillna(False).sum()) for k, v in fuera_de_rango.items()}, name="registros")

**Decisiones:**

- Un valor de 0 o negativo no describe la inversión: lo convertimos en faltante (no borramos el contrato, que sí existió).
- Una fecha de inicio o fin fuera de rango es casi siempre un error de digitación del año: la convertimos en faltante y la dejamos marcada.

In [ ]:
valor_invalido = fuera_de_rango["valor <= 0"].fillna(False)
df.loc[valor_invalido, "valor_del_contrato"] = np.nan

fechas_invalidas = 0
for c, regla in [("fecha_de_inicio_del_contrato", "inicio fuera de 2020-2035"),
                 ("fecha_de_fin_del_contrato", "fin fuera de 2020-2035")]:
    m = fuera_de_rango[regla].fillna(False)
    df.loc[m, c] = pd.NaT
    fechas_invalidas += int(m.sum())

registrar("T3 Estructurar", "Valores <= 0 convertidos a faltante", "Valores corregidos", valor_invalido.sum())
registrar("T3 Estructurar", "Fechas de inicio/fin fuera de rango convertidas a faltante",
          "Valores corregidos", fechas_invalidas)

#### Atípicos del valor

Los valores de contratos abarcan varios órdenes de magnitud (de cientos de miles a miles de millones de pesos), así que aplicamos la regla del rango intercuartílico al **logaritmo** del valor.

In [ ]:
log_valor = np.log10(df["valor_del_contrato"])
q1, q3 = log_valor.quantile([0.25, 0.75])
limite_inferior, limite_superior = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)

df["alerta_valor_atipico"] = ~log_valor.between(limite_inferior, limite_superior) & log_valor.notna()

pd.Series({"límite inferior (COP)": round(10 ** limite_inferior),
           "límite superior (COP)": round(10 ** limite_superior),
           "contratos atípicos": int(df["alerta_valor_atipico"].sum())})

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))

log_valor.plot.hist(bins=50, ax=ejes[0], edgecolor="white")
for limite in (limite_inferior, limite_superior):
    ejes[0].axvline(limite, color="firebrick", linestyle="--")
ejes[0].set(title="Distribución del valor de los contratos", xlabel="log10(valor en COP)", ylabel="Contratos")

ejes[1].boxplot(log_valor.dropna())
ejes[1].set(title="Diagrama de caja", xticks=[], ylabel="log10(valor en COP)")

plt.tight_layout()
plt.show()

**Decisión:** un contrato de licenciamiento nacional puede costar mil veces más que una capacitación municipal, y ambos son reales. Los atípicos **no se eliminan**: se marcan con `alerta_valor_atipico`. En la Parte 3 serán el insumo del modelo de detección de anomalías (Isolation Forest), que los evaluará junto con la modalidad y el tipo de servicio.

> La estadística *señala* candidatos; el conocimiento del dominio *decide*.

In [ ]:
registrar("T3 Estructurar", "Bandera de valor atípico (IQR sobre log10)", "Registros marcados",
          df["alerta_valor_atipico"].sum())

### T3.5 Variable derivada: tipo de servicio

La pregunta del proyecto distingue **qué** se compra. Creamos `tipo_servicio` a partir del objeto normalizado. Las reglas se evalúan **en orden** (con `np.select` gana la primera que se cumple), así que van de lo más específico a lo más general.

In [ ]:
REGLAS_SERVICIO = {
    "Respuesta": r"\bRESPUESTA A INCIDENTES\b|\bFORENSE|\bRECUPERACION\b|\bCONTENCION\b",
    "Detección": r"\bSOC\b|\bSIEM\b|\bMONITOREO\b|\bCENTRO DE OPERACIONES\b",
    "Preventivo": (r"\bAUDITORIA\b|\bPENTEST|\bPRUEBAS DE PENETRACION\b|\bETHICAL HACKING\b"
                   r"|\bVULNERABILIDAD|\bSGSI\b|\bISO ?27001\b|\bSENSIBILIZACION\b|\bCAPACITACION\b"
                   r"|\bDIAGNOSTICO\b|\bPOLITICA"),
    "Protección y licenciamiento": r"\bANTIVIRUS\b|\bEDR\b|\bFIREWALL|\bWAF\b|\bLICENCIA",
}

condiciones = [df["objeto_norm"].str.contains(p, regex=True, na=False) for p in REGLAS_SERVICIO.values()]
df["tipo_servicio"] = np.select(condiciones, list(REGLAS_SERVICIO), default="Otro / sin clasificar")

df["tipo_servicio"].value_counts(normalize=True).mul(100).round(1).to_frame("% de contratos")

Si el grupo "Otro / sin clasificar" es grande, lee una muestra y agrega reglas. Este es un buen lugar para el uso opcional de un modelo de lenguaje que permite la guía (etiquetar texto libre), **verificando** sus etiquetas contra una muestra revisada a mano.

In [ ]:
df.loc[df["tipo_servicio"] == "Otro / sin clasificar", "objeto_del_contrato"].head(10).to_frame()

In [ ]:
registrar("T3 Estructurar", "Tipo de servicio derivado del objeto", "Columnas creadas", 1,
          ", ".join(REGLAS_SERVICIO))

### T3.6 Variables categóricas

Las columnas con pocas categorías pasan a `category`: ocupan menos memoria y dejan claro que son cualitativas.

In [ ]:
columnas_category = columnas_categoricas + ["tipo_servicio", "departamento"]
memoria_antes = df.memory_usage(deep=True).sum() / 1e6

for c in columnas_category:
    df[c] = df[c].astype("category")

registrar("T3 Estructurar", "Columnas convertidas a category", "Columnas convertidas", len(columnas_category))
pd.Series({"memoria antes (MB)": round(memoria_antes, 2),
           "memoria después (MB)": round(df.memory_usage(deep=True).sum() / 1e6, 2)})

### T3.7 Código DIVIPOLA del municipio de la entidad

SECOP trae el municipio como **nombre** (`ciudad`), pero para cruzar con población, conectividad o cualquier fuente territorial necesitamos el código **DIVIPOLA de 5 dígitos** (Cali es `76001`). Lo obtenemos de la tabla oficial del DANE en datos.gov.co, también por API, y la cruzamos por **(departamento, municipio)**: el nombre solo no basta, porque hay municipios homónimos en distintos departamentos.

Primero descargamos la tabla y ubicamos sus columnas. En lugar de escribir sus nombres de memoria, los buscamos con un patrón: si el DANE cambia un nombre, el cuaderno lo detecta y avisa.

In [ ]:
ruta_divipola = RAW / "divipola_raw.parquet"
if ruta_divipola.exists():
    divipola_raw = pd.read_parquet(ruta_divipola)
else:
    divipola_raw = descargar(DATASET_DIVIPOLA, select="*")
    divipola_raw.to_parquet(ruta_divipola, index=False)

def buscar_columna(tabla, *patrones):
    for c in tabla.columns:
        if all(re.search(p, c, flags=re.IGNORECASE) for p in patrones):
            return c
    raise KeyError(f"No hay columna que cumpla {patrones}. Columnas: {list(tabla.columns)}")

col_cod_dpto = buscar_columna(divipola_raw, r"^cod", r"dpto|depart|_dp")
col_nom_dpto = buscar_columna(divipola_raw, r"^(?!cod)", r"dpto|depart")
col_cod_mpio = buscar_columna(divipola_raw, r"^cod", r"mpio|munic")
col_nom_mpio = buscar_columna(divipola_raw, r"^(?!cod)", r"mpio|munic")

print(col_cod_dpto, col_nom_dpto, col_cod_mpio, col_nom_mpio, sep=" | ")
divipola_raw.head()

Construimos la tabla de referencia. Si el código del municipio viene completo (5 dígitos) se usa tal cual; si trae solo los 3 del municipio, se antepone el del departamento. En ambos casos con `zfill`, que garantiza los ceros a la izquierda (`5001` → `05001`).

In [ ]:
cod_mpio = divipola_raw[col_cod_mpio].astype("string").str.replace(r"\D", "", regex=True)
cod_dpto = divipola_raw[col_cod_dpto].astype("string").str.replace(r"\D", "", regex=True).str.zfill(2)
divipola_codigo = cod_mpio.str.zfill(5).where(cod_mpio.str.len() > 3, cod_dpto + cod_mpio.str.zfill(3))

divipola = pd.DataFrame({
    "divipola": divipola_codigo,
    "departamento_norm": divipola_raw[col_nom_dpto].map(normalizar_texto),
    "ciudad_norm": divipola_raw[col_nom_mpio].map(normalizar_texto),
}).drop_duplicates()

assert divipola["divipola"].str.fullmatch(r"\d{5}").all()
divipola.head()

Los nombres oficiales y los de SECOP no siempre coinciden (`BOGOTA, D.C.` frente a `BOGOTA`). Quitamos la puntuación de ambos lados y aplicamos una tabla de **equivalencias**, que se amplía mirando la lista de lo que no cruzó.

In [ ]:
EQUIVALENCIAS_DPTO = {
    "BOGOTA D C": "BOGOTA", "DISTRITO CAPITAL DE BOGOTA": "BOGOTA", "BOGOTA DC": "BOGOTA",
    "ARCHIPIELAGO DE SAN ANDRES PROVIDENCIA Y SANTA CATALINA": "SAN ANDRES",
    "SAN ANDRES PROVIDENCIA Y SANTA CATALINA": "SAN ANDRES",
    "VALLE": "VALLE DEL CAUCA",
}
EQUIVALENCIAS_MPIO = {
    "BOGOTA D C": "BOGOTA", "BOGOTA DC": "BOGOTA", "SANTIAGO DE CALI": "CALI",
    "CARTAGENA DE INDIAS": "CARTAGENA", "SAN JOSE DE CUCUTA": "CUCUTA",
}

def llave_lugar(serie, equivalencias):
    sin_puntuacion = serie.str.replace(r"[^A-ZÑ0-9 ]", " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
    return sin_puntuacion.replace(equivalencias)

divipola["k_dpto"] = llave_lugar(divipola["departamento_norm"], EQUIVALENCIAS_DPTO)
divipola["k_mpio"] = llave_lugar(divipola["ciudad_norm"], EQUIVALENCIAS_MPIO)
df["k_dpto"] = llave_lugar(df["departamento_norm"].astype("string"), EQUIVALENCIAS_DPTO)
df["k_mpio"] = llave_lugar(df["ciudad_norm"].astype("string"), EQUIVALENCIAS_MPIO)

assert not divipola.duplicated(["k_dpto", "k_mpio"]).any(), "llave de lugar ambigua en DIVIPOLA"

Ahora el cruce, con `merge`. El parámetro `indicator=True` agrega la columna `_merge`, que dice si cada contrato encontró su municipio (`both`) o no (`left_only`). Con `validate="many_to_one"` `pandas` comprueba que a cada contrato le corresponda **a lo sumo un** municipio; si no, se detiene con un error en lugar de duplicar filas sin avisar.

In [ ]:
filas_antes = len(df)
df = df.merge(divipola[["k_dpto", "k_mpio", "divipola"]], on=["k_dpto", "k_mpio"],
              how="left", validate="many_to_one", indicator=True)
assert len(df) == filas_antes

df["_merge"].value_counts(normalize=True).mul(100).round(2).to_frame("% de contratos")

¿Por qué no cruzaron los que no cruzaron? Separamos los que **no tenían municipio** (faltante, lo tratamos en T4) de los que **tenían un nombre que no encontramos** (falta una equivalencia):

In [ ]:
sin_cruce = df[df["_merge"] == "left_only"]
pd.Series({
    "sin municipio en SECOP": int(sin_cruce["k_mpio"].isna().sum()),
    "con nombre que no cruzó": int(sin_cruce["k_mpio"].notna().sum()),
})

In [ ]:
sin_cruce[sin_cruce["k_mpio"].notna()].groupby(["k_dpto", "k_mpio"]).size().sort_values(ascending=False).head(15)

> Si la lista anterior no está vacía, agrega las equivalencias que falten y vuelve a ejecutar desde T3.7. El objetivo no es llegar al 100 %, sino **explicar** lo que no cruza.

In [ ]:
registrar("T3 Estructurar", "DIVIPOLA asignado por (departamento, municipio)", "Valores recodificados",
          (df["_merge"] == "both").sum(),
          f"{(df['_merge'] == 'both').mean() * 100:.1f} % de los contratos cruzaron")
df = df.drop(columns=["_merge", "k_dpto", "k_mpio", "departamento_norm", "ciudad_norm"])

### T3.8 Reglas de consistencia

| Regla | Lógica esperada |
|---|---|
| R1 | El contrato no empieza antes de firmarse |
| R2 | El contrato no termina antes de empezar |
| R3 | Un contrato de ciberseguridad tiene un objeto escrito |
| R4 | Toda entidad tiene NIT de 9 dígitos |

In [ ]:
reglas = {
    "R1 inicio antes de la firma": df["fecha_de_inicio_del_contrato"] < df["fecha_de_firma"],
    "R2 fin antes del inicio": df["fecha_de_fin_del_contrato"] < df["fecha_de_inicio_del_contrato"],
    "R3 objeto vacío": df["objeto_del_contrato"].isna(),
    "R4 NIT sin 9 dígitos": ~df["nit_entidad"].str.fullmatch(r"\d{9}").fillna(False),
}
pd.Series({nombre: int(m.fillna(False).sum()) for nombre, m in reglas.items()}, name="registros que violan")

**Decisión:** R1 y R2 no permiten saber cuál de las dos fechas está mal, así que no corregimos: creamos la bandera `alerta_fechas` y calculamos la duración solo cuando las fechas son coherentes. R4, si aparece, se revisa a mano: puede ser una entidad con NIT de otra longitud.

In [ ]:
df["alerta_fechas"] = (reglas["R1 inicio antes de la firma"] | reglas["R2 fin antes del inicio"]).fillna(False)

duracion = (df["fecha_de_fin_del_contrato"] - df["fecha_de_inicio_del_contrato"]).dt.days
df["duracion_dias"] = duracion.where(~df["alerta_fechas"]).astype("Int64")

registrar("T3 Estructurar", "Bandera de fechas incoherentes (R1, R2)", "Registros marcados", df["alerta_fechas"].sum())
registrar("T3 Estructurar", "Duración del contrato en días", "Columnas creadas", 1)
df["duracion_dias"].describe()

---
## T4. Completar los datos que faltan

| Tipo de faltante | Ejemplo en este caso | Tratamiento |
|---|---|---|
| **Deducible** de otra variable | Municipio de una entidad que en otros contratos sí lo reporta | Imputación lógica, con bandera |
| **Desconocido** | Valor del contrato faltante | No se inventa: se deja vacío |
| **No aplica** | Duración cuando las fechas son incoherentes | Se deja vacío a propósito |

In [ ]:
faltantes = df.isna().sum()
faltantes[faltantes > 0].to_frame("nulos").assign(pct=lambda t: (t["nulos"] / len(df) * 100).round(2))

### T4.1 Municipio de la entidad: imputación lógica

Una entidad tiene **una sede**. Si una entidad dejó el municipio en blanco en un contrato pero lo reportó en otros, el dato es **deducible**: usamos el DIVIPOLA más frecuente de ese NIT. Comprobamos primero que las entidades casi siempre reportan un único municipio:

In [ ]:
municipios_por_nit = df.groupby("nit_entidad")["divipola"].nunique()
municipios_por_nit.value_counts().sort_index().to_frame("entidades")

In [ ]:
divipola_antes = df["divipola"].copy()
df["divipola_imputado"] = df["divipola"].isna()

divipola_por_nit = df.groupby("nit_entidad")["divipola"].agg(
    lambda s: s.value_counts().index[0] if s.notna().any() else pd.NA)
df["divipola"] = df["divipola"].fillna(df["nit_entidad"].map(divipola_por_nit))
df["divipola_imputado"] = df["divipola_imputado"] & df["divipola"].notna()

pd.Series({"faltantes antes": int(divipola_antes.isna().sum()),
           "imputados desde otros contratos del mismo NIT": int(df["divipola_imputado"].sum()),
           "faltantes después": int(df["divipola"].isna().sum())})

In [ ]:
registrar("T4 Completar", "DIVIPOLA desde otros contratos del mismo NIT", "Valores imputados",
          df["divipola_imputado"].sum())

### T4.2 Faltantes que NO se completan

- `valor_del_contrato`: inventar un monto (por ejemplo, la mediana) cambiaría la suma de la inversión, que es justo lo que queremos medir.
- `divipola` que sigue vacío: la entidad nunca reportó municipio. En el análisis territorial quedará por fuera, y eso se declara.
- `duracion_dias` con fechas incoherentes: es un vacío a propósito.

### T4.3 Orden de las columnas

In [ ]:
bloques = {
    "identificacion": ["id_contrato", "urlproceso"],
    "entidad": ["nit_entidad", "nombre_entidad", "orden", "sector", "departamento", "ciudad",
                "divipola", "divipola_imputado"],
    "contrato": ["estado_contrato", "tipo_de_contrato", "modalidad_de_contratacion", "objeto_del_contrato",
                 "objeto_norm", "tipo_servicio", "unspsc", "unspsc_clase", "unspsc_segmento"],
    "proveedor": ["proveedor_adjudicado", "documento_proveedor"],
    "tiempo": ["fecha_de_firma", "fecha_de_inicio_del_contrato", "fecha_de_fin_del_contrato", "duracion_dias"],
    "valor": ["valor_del_contrato"],
    "banderas": ["alerta_valor_atipico", "alerta_fechas"],
}
orden_columnas = [c for grupo in bloques.values() for c in grupo if c in df.columns]

pd.Series({"columnas sin ubicar": [c for c in df.columns if c not in orden_columnas]})

In [ ]:
df = df[orden_columnas]
df.head()

---
## 2.5 Agregaciones de control

Antes de guardar, comprobamos que el conjunto limpio "cuenta una historia coherente". Estas tres agregaciones son además las funciones de `pandas` que la guía pide evidenciar (`groupby` con `agg`, `pivot_table` y `resample`). El análisis exploratorio completo es de la Entrega 2.

### Por orden de la entidad — `groupby` + `agg`

In [ ]:
(df.groupby("orden", observed=True)
   .agg(contratos=("id_contrato", "count"),
        entidades=("nit_entidad", "nunique"),
        valor_total_mm=("valor_del_contrato", lambda s: s.sum() / 1e6),
        valor_mediano_mm=("valor_del_contrato", lambda s: s.median() / 1e6))
   .round(1)
   .sort_values("valor_total_mm", ascending=False))

### Tipo de servicio por año — `pivot_table`

In [ ]:
pd.pivot_table(df.assign(anio=df["fecha_de_firma"].dt.year),
               index="tipo_servicio", columns="anio", values="id_contrato",
               aggfunc="count", fill_value=0, observed=True, margins=True, margins_name="Total")

### Contratos firmados por mes — `resample`

`resample` agrupa una serie indexada por fecha en intervalos regulares (`"ME"` = fin de mes). Es la base del análisis de estacionalidad y del evento de septiembre de 2023.

In [ ]:
mensual = df.set_index("fecha_de_firma")["id_contrato"].resample("ME").count()

ax = mensual.plot(figsize=(11, 4), marker=".")
ax.axvline(pd.Timestamp("2023-09-12"), color="firebrick", linestyle="--", label="Ataque a IFX Networks (12/09/2023)")
ax.set(title="Contratos de ciberseguridad firmados por mes", xlabel="", ylabel="Contratos")
ax.legend()
plt.show()

---
# 3. CARGAR: guardar el conjunto limpio

- **Parquet** conserva los tipos (fechas, categorías, booleanos): es el insumo de la Entrega 2.
- **CSV** con `utf-8-sig`, para abrirlo en Excel.

Ambos van a `data/processed/`, que **no** se sube al repositorio: quien clone el proyecto los regenera ejecutando este cuaderno.

In [ ]:
df.to_parquet(PROCESSED / "secop_ciberseguridad_limpio.parquet", index=False)
df.to_csv(PROCESSED / "secop_ciberseguridad_limpio.csv", index=False, encoding="utf-8-sig")

---
# 4. Validación final

Sobre el **archivo cargado**. Si la celda termina sin errores, todas las pruebas pasaron.

### 4.1 Llaves, duplicados y tipos

In [ ]:
final = pd.read_parquet(PROCESSED / "secop_ciberseguridad_limpio.parquet")

assert final.shape == df.shape
assert final["id_contrato"].is_unique
assert final.duplicated().sum() == 0

assert final["nit_entidad"].dropna().str.fullmatch(r"\d{9}").mean() > 0.99
assert final["divipola"].dropna().str.fullmatch(r"\d{5}").all()
assert final["unspsc"].dropna().str.fullmatch(r"\d{8}").all()

for c in columnas_fecha:
    assert pd.api.types.is_datetime64_any_dtype(final[c]), c
assert pd.api.types.is_float_dtype(final["valor_del_contrato"])
for c in ["alerta_valor_atipico", "alerta_fechas", "divipola_imputado"]:
    assert pd.api.types.is_bool_dtype(final[c]), c
for c in columnas_category:
    assert isinstance(final[c].dtype, pd.CategoricalDtype), c
for c in ["id_contrato", "nit_entidad", "divipola", "unspsc"]:
    assert pd.api.types.is_string_dtype(final[c]), c

### 4.2 Texto limpio, valores en rango y reglas

In [ ]:
for c in final.columns:
    if pd.api.types.is_string_dtype(final[c]) or isinstance(final[c].dtype, pd.CategoricalDtype):
        s = final[c].dropna().astype(str)
        assert (s == s.str.strip()).all(), c
        assert not s.str.contains(r"\s{2,}").any(), c
        assert not s.map(normalizar_texto).isin(MARCADORES).any(), c

assert (final["valor_del_contrato"].dropna() > 0).all()
assert final["fecha_de_firma"].dropna().between(FECHA_INICIO, FECHA_FIN).all()
assert not final["objeto_norm"].str.contains(EXCLUIR, regex=True, na=False).any()
assert set(final["tipo_servicio"].cat.categories) <= set(REGLAS_SERVICIO) | {"Otro / sin clasificar"}
assert (final["duracion_dias"].dropna() >= 0).all()
assert (final.groupby("nit_entidad")["nombre_entidad"].nunique() <= 1).all()

---
# 5. Reporte de calidad de datos

La guía pide: **filas antes y después**, **porcentaje de nulos por columna** y las **decisiones** con su justificación. Las tres salen de lo que fuimos registrando.

### 5.1 Antes y después

In [ ]:
pd.DataFrame({
    "original": [df_raw.shape[0], df_raw.shape[1], int(df_raw.isna().sum().sum()),
                 round(df_raw.memory_usage(deep=True).sum() / 1e6, 2)],
    "limpio": [final.shape[0], final.shape[1], int(final.isna().sum().sum()),
               round(final.memory_usage(deep=True).sum() / 1e6, 2)],
}, index=["filas", "columnas", "valores nulos", "memoria (MB)"])

### 5.2 Porcentaje de nulos por columna

Para las columnas originales comparamos el porcentaje "sin dato" (nulos + marcadores) de la descarga con el porcentaje de nulos del conjunto limpio.

In [ ]:
nulos_finales = (final.isna().mean() * 100).round(2)

reporte_nulos = pd.DataFrame({
    "pct_sin_dato_original": calidad_inicial["pct_sin_dato"],
    "pct_nulos_limpio": nulos_finales,
}).sort_values("pct_nulos_limpio", ascending=False)
reporte_nulos

### 5.3 Registro de decisiones

In [ ]:
registro = pd.DataFrame(cambios)
registro

In [ ]:
resumen = (registro.groupby("tipo", sort=False)["cantidad"].sum().to_frame("cantidad"))
resumen

In [ ]:
registro.to_csv(PROCESSED / "registro_limpieza_secop.csv", index=False, encoding="utf-8-sig")
reporte_nulos.to_csv(PROCESSED / "reporte_nulos_secop.csv", encoding="utf-8-sig")

---
## Conclusiones

1. **Documentar la fuente es parte del dato.** La ficha (entidad, enlace, fecha de corte, licencia) y el diccionario se generan desde los metadatos de la API y se guardan con la fecha de consulta.
2. **Filtrar en el servidor, decidir en `pandas`.** El filtro SoQL es amplio para no perder contratos; la regla fina, con límites de palabra y exclusiones, se ve y se justifica en el cuaderno.
3. **Definir la población es una decisión de limpieza.** Qué cuenta como "contrato de ciberseguridad" cambia todas las cifras posteriores; por eso se revisan muestras de lo incluido y de lo excluido.
4. **Las llaves se cuidan desde la Entrega 1.** NIT de 9 dígitos y DIVIPOLA de 5 dígitos, como texto, son las que permitirán integrar fuentes en la Entrega 2.
5. **Los atípicos y las incoherencias se marcan, no se borran.** Las banderas pasan a la siguiente etapa.
6. **No todo faltante se imputa.** El municipio se deduce del NIT; el valor faltante no se inventa.
7. **Validar sobre el archivo cargado** con `assert` convierte la calidad en algo comprobable.

## Ejercicios propuestos

1. **Precisión del filtro.** Toma 100 contratos conservados y 100 excluidos al azar, etiquétalos a mano y calcula la precisión y la exhaustividad de la regla de T1.3.
2. **Proveedores.** Normaliza `proveedor_adjudicado` y encuentra los 10 proveedores con mayor valor contratado. ¿Hay un mismo proveedor escrito de varias formas?
3. **Modalidad.** ¿Qué porcentaje del valor se contrató por contratación directa en cada año?
4. **Un año más.** Cambia `FECHA_FIN` a la fecha de hoy y vuelve a ejecutar todo el cuaderno. ¿Qué celdas tuviste que ajustar? Eso mide qué tan reproducible es tu proceso.